In [1]:
# import os
# from datasets import load_dataset

# dataset = load_dataset("PKU-Alignment/PKU-SafeRLHF")

# save_path = "./llm_distillation/datasets/hf/processed/pku_saferlhf"
# os.makedirs(f"{save_path}/train", exist_ok=True)
# os.makedirs(f"{save_path}/test",  exist_ok=True)

# dataset['train'].save_to_disk(f"{save_path}/train")
# dataset['test'].save_to_disk(f"{save_path}/test")

# print(f"Train: {len(dataset['train'])} | Test: {len(dataset['test'])}")

In [2]:
# # ── Key difference from multi-GPU: ──────────────────────────────────────────
# 1. distillation_config_enable_fsdp = False  → no torch.distributed init
# 2. distillation_config_pure_bf16   = True   → both models in bf16 to save VRAM
# 3. batch_size = 1, gradient_accumulation = 8 → effective batch = 8, fits on 1 GPU
# 4. rank = 0 hardcoded, no setup() call needed
# ─────────────────────────────────────────────────────────────────────────────
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
from configs import dataset as DATA_CONFIG
from configs import fsdp_config as FSDP_CONFIG
from configs import train_config as TRAIN_CONFIG
from configs import distillation_config as DISTIL_CONFIG
from train.train_utils import train
from configs.configs_utils import update_config
from data.data_utils import get_distillation_dataloader
from models.models_utils import get_distillation_models, get_optimizer
import torch
import random
args = {
    # ── Student (small model that gets trained) ───────────────────────────
    "model_name"                        : "Qwen/Qwen3-0.6B",

    # ── Teacher (frozen large model, provides soft targets) ───────────────
    # Use a smaller teacher for single-GPU testing — 1.1B fits alongside 410M
    "distillation_config_model_name"    : "Qwen/Qwen3-0.6B",

    # ── Dataset ───────────────────────────────────────────────────────────
    "dataset.file"                      : "./llm_distillation/datasets/loader/pku_saferlhf.py",

    # ── Training hyperparams (kept small for single-GPU test) ─────────────
    "lr"                                : 1e-5,
    "num_epochs"                        : 1,           # 1 epoch for testing
    "batch_size_training"               : 1,           # must be 1 on single GPU with teacher loaded
    "val_batch_size"                    : 1,
    "save_step"                         : 200,
    "f"                                 : 1,           # 1=fast OT, 2=greedy

    # ── Output ────────────────────────────────────────────────────────────
    "output_dir"                        : "./output_pku_distil_1gpu",

    # ── Distillation ON, FSDP OFF ─────────────────────────────────────────
    "distillation"                      : True,
    "distillation_config_enable_fsdp"   : False,   # ← critical: no distributed init
    "distillation_config_pure_bf16"     : True,    # ← bf16 for both models saves ~50% VRAM
    "distillation_config_distil_factor" : 1.5,
}

# ── Instantiate configs ───────────────────────────────────────────────────────
train_config  = TRAIN_CONFIG()
fsdp_config   = FSDP_CONFIG()
distil_config = DISTIL_CONFIG()
data_config   = DATA_CONFIG()

update_config((train_config, fsdp_config, data_config), **args)
update_config((distil_config), isSubmodule=True, **args)

# ── Manually set teacher name (mirrors finetuning.py line exactly) ────────────
distil_config.model_name = args["distillation_config_model_name"]

# ── Seeds ─────────────────────────────────────────────────────────────────────
torch.cuda.manual_seed(train_config.seed)
torch.manual_seed(train_config.seed)
random.seed(train_config.seed)

# ── rank = 0, no distributed setup needed (single GPU, no FSDP) ──────────────
rank = 0

print("Loading student + teacher models...")
student_tokenizer, teacher_tokenizer, model = get_distillation_models(
    train_config, distil_config, fsdp_config, rank, args
)
print(model)
print(f"\nVRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

/home/arya/aryaxai_labs/zera/Multi-Level-OT/models/checkpoint_handler.py:7: DeprecationWarning: `torch.distributed._shard.checkpoint` will be deprecated, use `torch.distributed.checkpoint` instead
  import torch.distributed._shard.checkpoint as dist_cp
/home/arya/miniforge3/envs/llm-stack/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/arya/aryaxai_labs/zera/Multi-Level-OT/models/tools.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


Loading student + teacher models...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 311/311 [00:00<00:00, 3915.90it/s]


--> Model Qwen/Qwen3-0.6B

--> Qwen/Qwen3-0.6B has 596.04992 Million params



Loading weights: 100%|██████████████████████| 311/311 [00:00<00:00, 2629.11it/s]


--> Model Qwen/Qwen3-0.6B

--> Qwen/Qwen3-0.6B has 596.04992 Million params

DistillationModel(
  (student): Qwen3ForCausalLM(
    (model): Qwen3Model(
      (embed_tokens): Embedding(151936, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          )
          (mlp): Qwen3MLP(
            (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (down_proj): Linear(in_features=3072, out_features=1024, bias=Fa

In [3]:
data_config.encoder_decoder = train_config.encoder_decoder

print("Building distillation dataloaders...")
(train_dl, teacher_train_dl,
 eval_dl,  teacher_eval_dl) = get_distillation_dataloader(
    data_config, train_config, distil_config,
    student_tokenizer, teacher_tokenizer, rank
)

print(f"Train batches : {len(train_dl)}")
print(f"Eval  batches : {len(eval_dl)}")

optimizer = get_optimizer(model, train_config, fsdp_config)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr           = train_config.lr,
    epochs           = train_config.num_epochs,
    steps_per_epoch  = len(train_dl),
    pct_start        = train_config.pct_start,
    div_factor       = train_config.div_factor,
    final_div_factor = train_config.final_div_factor,
)

print("Optimizer and scheduler ready.")

Building distillation dataloaders...


Tokenizing train: 100%|██████████| 73907/73907 [00:44<00:00, 1665.26 examples/s]


--> Training Set Length = 73907


Tokenizing validation: 100%|███████| 8211/8211 [00:05<00:00, 1486.62 examples/s]


--> Validation Set Length = 8211
--> Training Set Length = 73907
--> Validation Set Length = 8211
Train batches : 73907
Eval  batches : 8211
Optimizer and scheduler ready.


In [ ]:
# fsdp_config=None and local_rank=None because enable_fsdp=False
# This matches the exact branch in finetuning.py:
#   fsdp_config if train_config.enable_fsdp else None
#   local_rank  if train_config.enable_fsdp or distil_config.enable_fsdp else None

print("Starting single-GPU distillation training...")
results = train(
    model,
    train_dl,
    eval_dl,
    optimizer,
    scheduler,
    train_config.gradient_accumulation_steps,
    train_config,
    distil_config,
    data_config,
    teacher_train_dl,           # ← teacher dataloader (distillation)
    teacher_eval_dl,            # ← teacher eval dataloader
    None,                       # fsdp_config  — None, no FSDP
    None,                       # local_rank   — None, no FSDP
    rank,                       # 0
    train_config.f,             # OT method: 1=fast, 2=greedy
)

print("\n── Training Results ──")
for k, v in results.items():
    print(f"  {k}: {v}")